In [ ]:
# =============================================================================
# Supplementary Statistical Analyses
#
# PURPOSE:
# This notebook performs supplementary analyses examining relationships between
# speech/pause metrics and standardized participant measures.
#
# INPUTS:
# - CWNS_CWS_all_details.csv
# - CWNS_CWS_all_demo_and_score_details.csv
#
# OUTPUTS:
# - Supplementary p-value table: pval_supp.csv
# - Partial correlation / supplementary association tables
#
# ANALYSIS OVERVIEW:
# 1. Load participant-level speech/pause metrics
# 2. Load demographic and standardized score data
# 3. Harmonize participant-level datasets
# 4. Convert selected timing variables to milliseconds
# 5. Compute supplementary associations and partial correlations
#
# NOTE:
# These analyses support the main group-level models but are not the primary
# analysis pipeline.
# =============================================================================

In [ ]:
import os
import pandas as pd
import statsmodels.api as sm
from scipy.stats import pearsonr

In [ ]:
# Project directory structure
#
# ATAS_CWNS_CWS_Project/        ← THIS is base_dir
# │
# ├── Audio_files_clipped/      ← create this folder; clipped audio files stored here
# ├── Visualize/                ← create this folder; .wav audio files for visualization/analysis stored here
# ├── Stat_csv_files/           ← input CSV files and summary statistics CSV files stored here
# ├── Individual_OutputCSV_Files/ ← individual output CSV files stored here
# │
# ├── Notebooks/                ← notebooks stored here
# │   ├── ATAS_Visualization/   ← visualization notebook
# │   └── ML_Analysis/          ← machine learning analysis notebooks
# │
# └── Scripts/                  ← function notebooks
#
base_dir = '..../ATAS_CWNS_CWS_Project'  # Change to required project directory
os.chdir(base_dir)  # Set the working directory

# Read the CSV file
xdata_pre = pd.read_csv(base_dir + "/Stat_csv_files/CWNS_CWS_all_details.csv") # Assuming data.csv exists in the specified directory

# Load participant demographic and standardized score data
csv_file = base_dir+"/Stat_csv_files/CWNS_CWS_all_demo_and_score_details.csv"  
scores_file = pd.read_csv(csv_file)

# for storing the p-value file
file_path = os.path.join(base_dir, "Stat_csv_files", "pval_supp.csv") # Define output path for supplementary p-values

In [ ]:
# Rename columns to make variable names easier to use in formulas and tables
xdata_pre = xdata_pre.rename(columns={
    'Mean Pause_s': 'Mean_Pause_s',
    'Mean Speech_s': 'Mean_Speech_s',
    'CV Pause': 'CV_Pause',
    'CV Speech': 'CV_Speech',
})

# Convert duration variables from seconds to milliseconds for easier interpretation
# List of columns to convert
columns_to_convert = ['Mean_Speech_s', 'Mean_Pause_s', 'long_p_durations_mean', 'short_p_durations_mean']

# Convert from seconds to milliseconds and create new columns
for col in columns_to_convert:
    xdata_pre[f"{col}_ms"] = xdata_pre[col] * 1000

In [ ]:
# Define variables for continuous and count models
continuous_metrics = {
    "Speech_Rate": "Speech Rate",
    "Pause_Duration_s": "Pause Duration (s)",
    "Mean_Pause_s_ms": "Mean Pause Duration (ms)",
    "Mean_Speech_s_ms": "Mean Speech Duration (ms)",
    "CV_Pause": "Coefficient of Variation - Pause",
    "CV_Speech": "Coefficient of Variation - Speech",
    "long_p_durations_mean_ms": "Mean Long Pause Duration (ms)",
    "short_p_durations_mean_ms": "Mean Short Pause Duration (ms)",
    "long_p_durations_cv": "CV of Long Pauses",
    "short_p_durations_cv": "CV of Short Pauses"
}

count_metrics = {
    "Pause_Events": "Number of Pause Events",
    "long_p_count": "Long Pause Events",
    "short_p_count": "Short Pause Events"
}

In [ ]:
def custom_round(x):
    if pd.isna(x):
        return x
    return int(x + 0.5)
# Apply to Age column
xdata_pre_1 = xdata_pre.copy()
xdata_pre_1['Age'] = xdata_pre_1['Age'].apply(custom_round)

In [ ]:
# --------------------------------------------------
# 1. Make copies so original dataframes stay unchanged
# --------------------------------------------------
xdata_pre1 = xdata_pre_1.copy()
score_file = scores_file.copy()
# --------------------------------------------------
# 2. Clean the merge key
# --------------------------------------------------
xdata_pre1["ID"] = xdata_pre1["ID"].astype(str).str.strip()
score_file["ID"] = score_file["ID"].astype(str).str.strip()

# --------------------------------------------------
# 3. Select only the columns you want to bring in
#    Edit these names if your actual column names differ
# --------------------------------------------------
cols_to_add = ["ID", "PPVT", "EVT", "ses_edu"]

missing_cols = [c for c in cols_to_add if c not in score_file.columns]
if missing_cols:
    raise ValueError(f"These columns are missing from score_file: {missing_cols}")

# keep one row per ID in score_file
score_subset = score_file[cols_to_add].drop_duplicates(subset="ID")

# optional: check for duplicated IDs in score_file
dup_ids = score_file.loc[score_file.duplicated(subset="ID", keep=False), "ID"].unique()
if len(dup_ids) > 0:
    print("Warning: duplicated IDs found in score_file. Using first unique row per ID.")
    print(dup_ids)

# --------------------------------------------------
# 4. Merge into xdata_pre1
# --------------------------------------------------
xdata = xdata_pre1.merge(score_subset, on="ID", how="left", validate="m:1")

# --------------------------------------------------
# 5. Quick checks
# --------------------------------------------------
print("xdata shape:", xdata.shape)
print("\nMissing values after merge:")
print(xdata[["PPVT", "EVT", "ses_edu"]].isna().sum())

print("\nPreview:")
display(xdata.head())

In [ ]:
# Compute supplementary associations between speech/pause metrics
# and standardized scores while accounting for covariates
key_metrics = {
    "Speech_Rate": "Speech rate",
    "Pause_Duration_s": "Pause duration (s)",
    "Mean_Speech_s_ms": "Mean speech duration (ms)",
    "Pause_Events": "Number of pause events",
    "long_p_count": "Long pause events",
    "short_p_count": "Short pause events"
}

df = xdata.copy()

df["Group"] = df["Group"].astype(str).str.strip().replace({
    "cws": "CWS", "cwS": "CWS", "CWS": "CWS",
    "cwns": "CWNS", "Cwns": "CWNS", "CWNS": "CWNS"
})

df["Sex"] = df["Sex"].astype(str).str.strip().replace({
    "m": "M", "male": "M", "M": "M",
    "f": "F", "female": "F", "F": "F"
})

for col in ["Age", "PPVT", "EVT"] + list(key_metrics):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

def format_p_value(p):
    if pd.isna(p):
        return "—"
    if p < .001:
        return "< .001"
    return f"{p:.3f}".lstrip("0")

def format_number(x, decimals=2):
    return "—" if pd.isna(x) else f"{x:.{decimals}f}"

def effect_size_label(r):
    if pd.isna(r):
        return "—"
    r = abs(r)
    if r < 0.10:
        return "Negligible"
    if r < 0.30:
        return "Small"
    if r < 0.50:
        return "Moderate"
    return "Large"

def residualize(y, covars):
    X = sm.add_constant(covars, has_constant="add")
    return sm.OLS(y, X).fit().resid

def build_partial_corr_table(data, covariates=("PPVT", "EVT")):
    rows = []

    for covariate in covariates:
        for metric_code, metric_label in key_metrics.items():
            d = data[[metric_code, covariate, "Group", "Sex", "Age"]].dropna().copy()
            n = len(d)

            if n < 10:
                rows.append({
                    "Language covariate": covariate,
                    "ATAS metric": metric_label,
                    "N": n,
                    "Partial correlation, r": "—",
                    "p value": "—",
                    "Effect size": "Too few observations"
                })
                continue

            try:
                controls = pd.get_dummies(
                    d[["Group", "Sex", "Age"]],
                    columns=["Group", "Sex"],
                    drop_first=True
                ).astype(float)

                y_resid = residualize(d[metric_code].astype(float), controls)
                x_resid = residualize(d[covariate].astype(float), controls)

                r, p = pearsonr(x_resid, y_resid)

                rows.append({
                    "Language covariate": covariate,
                    "ATAS metric": metric_label,
                    "N": n,
                    "Partial correlation, r": format_number(r, 2),
                    "p value": format_p_value(p),
                    "Effect size": effect_size_label(r)
                })

            except Exception:
                rows.append({
                    "Language covariate": covariate,
                    "ATAS metric": metric_label,
                    "N": n,
                    "Partial correlation, r": "—",
                    "p value": "—",
                    "Effect size": "Failed"
                })

    return pd.DataFrame(rows)

table_partial_r = build_partial_corr_table(df)
table_partial_r.to_csv(file_path, index=False)

print(table_partial_r)